# Limpeza de `atividades.parquet`

**Objetivo:** gerar a tabela de treinos pronta para análise em `dados/processados/atividades_limpa.parquet`.

**Regras aplicadas** (todas registradas no `decisoes.md` em 31/10/2026; o diagnóstico está no notebook 04):

| Passo | Regra |
|---|---|
| 1 | Tirar o treino gravado duas vezes com `atividade_id` diferente (fica o menor id) |
| 2 | Converter o `inicio` do Garmin de UTC para Brasília (−3h); criar `data` |
| 3 | Traduzir os tipos em inglês (run, walk, ride, strength, hiit) para o português |
| 4 | Tirar o registro do app quando o mesmo treino veio também do relógio (fica o do relógio) |
| 5 | Recalcular a distância incoerente como duração ÷ ritmo (`distancia_recalculada`) |
| 6 | Duração 0 vira ausente, sem tirar o treino (`duracao_zero`); marcar os que não têm outra prova (`sem_outra_prova`) |
| 7 | Falha de GPS: distância e ritmo continuam ausentes, sem imputação; `gps_ok = False` é a marca |

**Ficam como estão, sem decisão registrada:** dois treinos diferentes da mesma conta no mesmo minuto e os treinos com
mais de 5h (plausíveis: pedal longo, ultra).

**Atenção ao usar:** `inicio` e `data` estão em horário de Brasília para todas as origens. Minutos se somam só com
`duracao_min` não ausente. Para comparar 2025 (só em `uso_mensal.csv`) com 2026, usar só a origem `app` (decisão de 31/10).

Nada em `dados/brutos/` é alterado: o hash dos arquivos é conferido no início e no fim.
As linhas antes e depois de cada passo vão para `saidas/limpeza_atividades_registro.csv`.

In [1]:
import hashlib
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
BRUTOS = ROOT / "dados" / "brutos"
PROCESSADOS = ROOT / "dados" / "processados"
SAIDAS = ROOT / "saidas"
CORTE = pd.Timestamp("2026-09-30")
FIM_CORTE = CORTE + pd.Timedelta(days=1)   # 01/10/2026 00:00, horário de Brasília

ARQUIVOS = ["atividades.parquet"]

def sha1(path):
    return hashlib.sha1(path.read_bytes()).hexdigest()

hash_inicio = {f: sha1(BRUTOS / f) for f in ARQUIVOS}

registro = []

def registrar(passo, regra, df, alteradas=0):
    anterior = registro[-1]["linhas"] if registro else len(df)
    registro.append({"passo": passo, "regra": regra, "linhas": len(df), "removidas": anterior - len(df),
                     "alteradas": int(alteradas)})
    print(f"{passo}: {len(df):,} linhas ({anterior - len(df):,} removidas, {int(alteradas):,} alteradas)")
    return df

## 0. Leitura

O parquet mantém os tipos gravados: números como `float`, `gps_ok` como `bool`, `inicio` como texto sem fuso.

In [2]:
ativ = pd.read_parquet(BRUTOS / "atividades.parquet")
assert len(ativ) == 1_254_475   # cabeçalho da seção 09 do dicionário
assert ativ["atividade_id"].is_unique
COLUNAS_BRUTO = list(ativ.columns)
print(ativ.dtypes.astype(str).to_dict())

base = registrar("0. Arquivo bruto", "atividades.parquet sem filtro", ativ.copy())

{'atividade_id': 'object', 'user_id': 'object', 'inicio': 'object', 'tipo': 'object', 'duracao_min': 'float64', 'distancia_km': 'float64', 'ritmo_min_km': 'float64', 'fc_media': 'float64', 'origem': 'object', 'gps_ok': 'bool', 'treino_com': 'object'}
0. Arquivo bruto: 1,254,475 linhas (0 removidas, 0 alteradas)


## 1. Treino gravado duas vezes

Todas as colunas iguais, menos `atividade_id`. Fica o menor id.

In [3]:
base = base.sort_values("atividade_id", kind="stable")
copia = base.duplicated(subset=[c for c in COLUNAS_BRUTO if c != "atividade_id"], keep="first")
assert copia.sum() == 1
base = registrar("1. Sem cópia exata", "todas as colunas iguais menos atividade_id; fica o menor id",
                 base.loc[~copia].copy())

1. Sem cópia exata: 1,254,474 linhas (1 removidas, 0 alteradas)


## 2. Horário em Brasília

O Garmin grava o `inicio` em UTC sem o "Z"; app e Apple Health gravam em horário de Brasília. Com −3h no Garmin, a
distribuição por hora fica igual à do app. O texto original não é usado para datas.

In [4]:
texto = pd.to_datetime(base["inicio"], format="%Y-%m-%d %H:%M:%S")
eh_garmin = base["origem"] == "garmin"
base["inicio"] = texto.where(~eh_garmin, texto - pd.Timedelta(hours=3))
base["data"] = base["inicio"].dt.normalize()

mudou_dia = (base["data"] != texto.dt.normalize())
print(f"convertidos (Garmin): {eh_garmin.sum():,}; mudam de dia: {mudou_dia.sum():,}; "
      f"pareceriam depois do corte pelo texto: {(texto >= FIM_CORTE).sum()}")
print(f"inicio em Brasília: {base['inicio'].min()} a {base['inicio'].max()}")
assert base["inicio"].notna().all()
assert (base["inicio"] < FIM_CORTE).all()

base = registrar("2. Horário em Brasília", "inicio do Garmin -3h; data = dia em Brasília", base, eh_garmin.sum())

convertidos (Garmin): 207,954; mudam de dia: 5,828; pareceriam depois do corte pelo texto: 34
inicio em Brasília: 2026-01-01 05:00:00 a 2026-09-30 21:59:00
2. Horário em Brasília: 1,254,474 linhas (0 removidas, 207,954 alteradas)


## 3. Tipos em português

A partir de 15/04/2026 o app grava cinco tipos em inglês. Cada par tem a mesma mediana e o mesmo p90 de duração,
distância, ritmo e FC (notebook 04).

In [5]:
TIPO_PT = {"run": "corrida", "walk": "caminhada", "ride": "pedal", "strength": "forca", "hiit": "funcional"}
em_ingles = base["tipo"].isin(TIPO_PT)
print("em inglês:", base.loc[em_ingles, "tipo"].value_counts().to_dict())
assert (base.loc[em_ingles, "origem"] == "app").all()
assert base.loc[em_ingles, "data"].min() >= pd.Timestamp("2026-04-15")

base["tipo"] = base["tipo"].replace(TIPO_PT)
TIPOS = {"corrida", "caminhada", "pedal", "funcional", "forca", "yoga"}
assert set(base["tipo"]) == TIPOS, set(base["tipo"])
display(base["tipo"].value_counts().to_frame("treinos"))

base = registrar("3. Tipos em português", "run, walk, ride, strength, hiit traduzidos", base, em_ingles.sum())

em inglês: {'run': 162108, 'hiit': 109934, 'walk': 88509, 'strength': 85547, 'ride': 40765}


,treinos
tipo,
corrida,385926
funcional,255410
caminhada,206533
forca,198251
yoga,113849
pedal,94505


3. Tipos em português: 1,254,474 linhas (0 removidas, 486,863 alteradas)


## 4. Mesmo treino pelo app e pelo relógio

Mesma conta, mesmo horário em Brasília, mesmo tipo e origens diferentes. Fica o registro do relógio (Garmin ou
Apple Health), que é a fonte do treino; sai o do app.

**Contagem diferente da registrada (1.990):** o diagnóstico contou as "linhas a mais" nos grupos de mesmo horário. Em 6
grupos há três registros (app + dois do relógio): o do app repete um dos registros do relógio e o terceiro é outro treino,
que fica. E 2 pares app + relógio do mesmo tipo estavam em grupos com um terceiro treino de outro tipo. Aplicando a regra
par a par, saem 1.986 registros do app, um por par.

In [6]:
CHAVE = ["user_id", "inicio", "tipo"]
cand = base.loc[base.duplicated(CHAVE, keep=False), CHAVE + ["origem"]].copy()
cand["eh_app"] = cand["origem"] == "app"
cand["eh_relogio"] = ~cand["eh_app"]
grupo = cand.groupby(CHAVE)
par = grupo["eh_app"].transform("max") & grupo["eh_relogio"].transform("max")
sai_idx = cand.index[par & cand["eh_app"]]
sai = base.index.isin(sai_idx)

combinacoes = cand.loc[par].groupby(CHAVE)["origem"].agg(lambda s: " + ".join(sorted(s))).value_counts()
display(combinacoes.to_frame("grupos"))
apps_por_grupo = cand.loc[par & cand["eh_app"]].groupby(CHAVE).size()
print(f"grupos (conta, horário, tipo) com app e relógio: {par.groupby([cand[c] for c in CHAVE]).any().sum():,}; saem (app): {sai.sum():,}")
assert sai.sum() == 1_986
assert (apps_por_grupo == 1).all()   # um registro do app por grupo; os do relógio ficam todos

removidos_app = base.loc[sai].copy()   # usados nas conferências dos passos 5 a 7
base = registrar("4. Sem duplicata app + relógio", "mesma conta, horário e tipo; sai o registro do app, fica o do relógio",
                 base.loc[~sai].copy())

,grupos
origem,
app + garmin,1064
app + apple_health,916
app + garmin + garmin,4
app + apple_health + apple_health,2


grupos (conta, horário, tipo) com app e relógio: 1,986; saem (app): 1,986


4. Sem duplicata app + relógio: 1,252,488 linhas (1,986 removidas, 0 alteradas)


## 5. Distância incoerente

Distância maior que o dobro de duração ÷ ritmo, com duração > 0 (observado: 105 a 10.013 km; 999 no bruto, menos os
que já saíram no passo 4). Nos demais treinos as
três colunas batem (razão entre 0,996 e 1,004). A distância é recalculada com 2 casas, como no arquivo.

In [7]:
esperada = base["duracao_min"] / base["ritmo_min_km"]
absurda = (base["duracao_min"] > 0) & (base["distancia_km"] > 2 * esperada)
print(f"distâncias incoerentes: {absurda.sum():,}; de {base.loc[absurda, 'distancia_km'].min():,.0f} a "
      f"{base.loc[absurda, 'distancia_km'].max():,.0f} km")
esperada_rem = removidos_app["duracao_min"] / removidos_app["ritmo_min_km"]
absurda_rem = int(((removidos_app["duracao_min"] > 0) & (removidos_app["distancia_km"] > 2 * esperada_rem)).sum())
print(f"no bruto eram 999: {absurda_rem} estavam em registros do app que saíram no passo 4 como duplicata")
assert absurda.sum() + absurda_rem == 999

base["distancia_recalculada"] = absurda
base.loc[absurda, "distancia_km"] = esperada[absurda].round(2)
print("depois:", base.loc[absurda, "distancia_km"].describe().round(2).to_dict())

base = registrar("5. Distância recalculada", "distancia_km = duracao_min / ritmo_min_km quando passa do dobro", base,
                 absurda.sum())

distâncias incoerentes: 997; de 105 a 10,013 km
no bruto eram 999: 2 estavam em registros do app que saíram no passo 4 como duplicata
depois: {'count': 997.0, 'mean': 9.96, 'std': 11.82, 'min': 1.8, '25%': 4.55, '50%': 6.56, '75%': 9.71, 'max': 160.45}
5. Distância recalculada: 1,252,488 linhas (0 removidas, 997 alteradas)


## 6. Duração 0

O treino continua contando, mas a duração vira ausente (fora de somas e médias de minutos). `sem_outra_prova` marca os
que também não têm distância > 0 nem FC, para rodar as análises com e sem eles.

In [8]:
zero = base["duracao_min"] == 0
outra_prova = (base["distancia_km"] > 0) | base["fc_media"].notna()
base["duracao_zero"] = zero
base["sem_outra_prova"] = zero & ~outra_prova
base.loc[zero, "duracao_min"] = float("nan")
print(f"duração 0: {zero.sum():,}; com outra prova: {(zero & outra_prova).sum():,}; sem outra prova: {base['sem_outra_prova'].sum():,}")
print("sem outra prova, por tipo:", base.loc[base["sem_outra_prova"], "tipo"].value_counts().to_dict(),
      "| origem:", base.loc[base["sem_outra_prova"], "origem"].value_counts().to_dict())
zero_rem = int((removidos_app["duracao_min"] == 0).sum()) + int((ativ.loc[copia.index[copia], "duracao_min"] == 0).sum())
print(f"no bruto eram 3.649 com duração 0: {zero_rem} saíram nos passos 1 e 4")
assert zero.sum() + zero_rem == 3_649

base = registrar("6. Duração 0 como ausente", "duracao_min 0 vira ausente; duracao_zero e sem_outra_prova", base, zero.sum())

duração 0: 3,645; com outra prova: 2,647; sem outra prova: 998
sem outra prova, por tipo: {'funcional': 435, 'forca': 328, 'yoga': 220, 'corrida': 9, 'caminhada': 4, 'pedal': 2} | origem: {'app': 998}
no bruto eram 3.649 com duração 0: 4 saíram nos passos 1 e 4
6. Duração 0 como ausente: 1,252,488 linhas (0 removidas, 3,645 alteradas)


## 7. Falha de GPS

Distância e ritmo ausentes ficam ausentes. A marca já existe: `gps_ok = False`. A célula só confere o padrão do
diagnóstico (só app, só treinos ao ar livre, de 04/05 a 28/06/2026).

In [9]:
falhou = ~base["gps_ok"]
print(f"gps_ok = False: {falhou.sum():,} treinos, {base.loc[falhou, 'user_id'].nunique():,} contas; "
      f"de {base.loc[falhou, 'data'].min().date()} a {base.loc[falhou, 'data'].max().date()}")
print("sem distância:", base.loc[falhou, "distancia_km"].isna().sum(), "| sem ritmo:", base.loc[falhou, "ritmo_min_km"].isna().sum())
assert (base.loc[falhou, "origem"] == "app").all()
assert set(base.loc[falhou, "tipo"]) <= {"corrida", "caminhada", "pedal"}
sem_gps_ok = base["gps_ok"] & base["tipo"].isin({"corrida", "caminhada", "pedal"}) & base["distancia_km"].isna()
assert sem_gps_ok.sum() == 0   # todo treino ao ar livre sem distância é falha de GPS

base = registrar("7. Falha de GPS mantida", "distância e ritmo ausentes, sem imputação; gps_ok = False é a marca", base)

gps_ok = False: 14,778 treinos, 4,319 contas; de 2026-05-04 a 2026-06-28
sem distância: 7324 | sem ritmo: 14778
7. Falha de GPS mantida: 1,252,488 linhas (0 removidas, 0 alteradas)


## 8. Conferências

- 1.252.488 treinos (1.254.475 − 1 cópia − 1.986 duplicatas app + relógio), `atividade_id` único;
- toda conta e todo parceiro de treino (`treino_com`) existem em `usuarios_limpa.parquet`;
- nenhum treino antes do dia do cadastro nem depois do corte;
- nenhuma duplicata app + relógio sobrando; distância, duração e ritmo coerentes onde os três existem.

In [10]:
usuarios = pd.read_parquet(PROCESSADOS / "usuarios_limpa.parquet", columns=["user_id", "data_cadastro"]).set_index("user_id")
base["treino_com"] = base["treino_com"].where(base["treino_com"] != "", None)

tres = base["duracao_min"].notna() & base["distancia_km"].notna() & base["ritmo_min_km"].notna()
razao = base.loc[tres, "distancia_km"] / (base.loc[tres, "duracao_min"] / base.loc[tres, "ritmo_min_km"])
sobra = base.duplicated(CHAVE, keep=False)
sobra_grupo = base.loc[sobra].groupby(CHAVE).agg(app=("origem", lambda s: (s == "app").any()), relogio=("origem", lambda s: (s != "app").any()))

conferencias = {
    "treinos na tabela limpa": (len(base), 1_252_488),
    "atividade_id repetido": (int(base["atividade_id"].duplicated().sum()), 0),
    "contas fora de usuarios_limpa": (int((~base["user_id"].isin(usuarios.index)).sum()), 0),
    "treino_com fora de usuarios_limpa": (int((base["treino_com"].notna() & ~base["treino_com"].isin(usuarios.index)).sum()), 0),
    "treino antes do dia do cadastro": (int((base["data"] < base["user_id"].map(usuarios["data_cadastro"])).sum()), 0),
    "treino depois do corte": (int((base["data"] > CORTE).sum()), 0),
    "grupos app + relógio, mesmo tipo, sobrando": (int((sobra_grupo["app"] & sobra_grupo["relogio"]).sum()), 0),
    "distância fora de 0,99-1,01 x duração/ritmo": (int((~razao.between(0.99, 1.01)).sum()), 0),
    "tipos fora dos seis": (len(set(base["tipo"]) - TIPOS), 0),
}
tabela = pd.DataFrame(conferencias, index=["obtido", "esperado"]).T
tabela["ok"] = tabela["obtido"] == tabela["esperado"]
display(tabela)
assert tabela["ok"].all()
print(f"treinos diferentes da mesma conta no mesmo horário e tipo (ficam, sem decisão): {len(sobra_grupo):,} grupos")

,obtido,esperado,ok
treinos na tabela limpa,1252488,1252488,True
atividade_id repetido,0,0,True
contas fora de usuarios_limpa,0,0,True
treino_com fora de usuarios_limpa,0,0,True
treino antes do dia do cadastro,0,0,True
treino depois do corte,0,0,True
"grupos app + relógio, mesmo tipo, sobrando",0,0,True
"distância fora de 0,99-1,01 x duração/ritmo",0,0,True
tipos fora dos seis,0,0,True


treinos diferentes da mesma conta no mesmo horário e tipo (ficam, sem decisão): 584 grupos


## 9. Saídas e integridade dos brutos

In [11]:
COLUNAS = ["atividade_id", "user_id", "inicio", "data", "tipo", "duracao_min", "distancia_km", "ritmo_min_km", "fc_media",
           "origem", "gps_ok", "treino_com", "distancia_recalculada", "duracao_zero", "sem_outra_prova"]
limpa = base.sort_values(["user_id", "inicio", "atividade_id"], kind="stable")[COLUNAS].reset_index(drop=True)

PROCESSADOS.mkdir(parents=True, exist_ok=True)
destino = PROCESSADOS / "atividades_limpa.parquet"
limpa.to_parquet(destino, index=False)

relida = pd.read_parquet(destino)
pd.testing.assert_frame_equal(relida, limpa)

pd.DataFrame(registro).to_csv(SAIDAS / "limpeza_atividades_registro.csv", index=False)

print(f"{destino.relative_to(ROOT)}: {len(limpa):,} linhas x {limpa.shape[1]} colunas")
display(limpa.dtypes.rename("tipo").to_frame().assign(ausentes=limpa.isna().sum()))
display(pd.DataFrame(registro))

hash_fim = {f: sha1(BRUTOS / f) for f in ARQUIVOS}
assert hash_fim == hash_inicio
print("dados/brutos/ inalterado:", hash_fim == hash_inicio)

dados\processados\atividades_limpa.parquet: 1,252,488 linhas x 15 colunas


,tipo,ausentes
atividade_id,object,0
user_id,object,0
inicio,datetime64[ns],0
data,datetime64[ns],0
tipo,object,0
duracao_min,float64,3645
distancia_km,float64,573964
ritmo_min_km,float64,581418
fc_media,float64,764986
origem,object,0


,passo,regra,linhas,removidas,alteradas
0,0. Arquivo bruto,atividades.parquet sem filtro,1254475,0,0
1,1. Sem cópia exata,todas as colunas iguais menos atividade_id; fi...,1254474,1,0
2,2. Horário em Brasília,inicio do Garmin -3h; data = dia em Brasília,1254474,0,207954
3,3. Tipos em português,"run, walk, ride, strength, hiit traduzidos",1254474,0,486863
4,4. Sem duplicata app + relógio,"mesma conta, horário e tipo; sai o registro do...",1252488,1986,0
5,5. Distância recalculada,distancia_km = duracao_min / ritmo_min_km quan...,1252488,0,997
6,6. Duração 0 como ausente,duracao_min 0 vira ausente; duracao_zero e sem...,1252488,0,3645
7,7. Falha de GPS mantida,"distância e ritmo ausentes, sem imputação; gps...",1252488,0,0


dados/brutos/ inalterado: True
